# SR Local Explanation: single input-output pair

This notebook explains **one specific prediction** from a symbolic regression model  

## 1. Setup

In [1]:
import sys
from pathlib import Path

_root = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "sr_local.py").exists())
sys.path.insert(0, str(_root))

from sr_local import (
    evaluate_expression_at_input,
    generate_local_facts,
    generate_local_plain_language_prompt,
)

## 2. Inputs — change these to test a different sample

- `EXPR` is the symbolic regression expression  
- `SAMPLE_INPUT` is the one data point you want to explain.  Keys can be **x_N names** or **display names**, both work via `VAR_MAP`  
- `MODEL_CONTEXT` is optional free-form metadata you want the LLM to have access to

In [2]:
EXPR = "((x_10**3 * x_7) + (x_14 - x_7))"

VAR_MAP = {
    "x_7":  "MutationTaster_score",
    "x_10": "MutPred_score",
    "x_14": "VEST4_score",
}

# Input using x_N keys
SAMPLE_INPUT_XN = {
    "x_10": 0.75,   # MutPred_score
    "x_7":  0.82,   # MutationTaster_score
    "x_14": 0.61,   # VEST4_score
}

# Same input using display names
SAMPLE_INPUT_NAMED = {
    "MutPred_score":        0.75,
    "MutationTaster_score": 0.82,
    "VEST4_score":          0.61,
}

# Optional: output value from the GP evaluation or from another tool,
# to compare against the symbolic re-evaluation here
PROVIDED_OUTPUT = None # 4.10   # set to None to skip comparison

# Optional metadata for the LLM (not verified computationally)
MODEL_CONTEXT = {
    "model_source": "GP run, generation 200, fitness = 0.91",
    "target_variable": "pathogenicity_score  (0 = benign, 1 = pathogenic)",
    "domain_annotations": {
        "MutPred_score":        "Probability of pathogenicity predicted by MutPred2",
        "MutationTaster_score": "Disease-causing probability from MutationTaster",
        "VEST4_score":          "Functional impact score from VEST4",
    },
}

## 3. `evaluate_expression_at_input`

Parses and simplifies the expression, then substitutes the provided input values

Handles:
- **x_N keys** or **display names** (resolved via `var_map`)
- **missing variables**, reports which ones are absent and skips evaluation
- **non-finite results**, singularities (`zoo`), overflow (`oo`) are caught and reported as warnings

Returns a flat dict of verified facts; no interpretation is added

In [3]:
# --- using x_N keys ---
ev_xn = evaluate_expression_at_input(EXPR, SAMPLE_INPUT_XN, VAR_MAP)

print("Expression (simplified):", ev_xn["simplified"])
print("Expression (named)     :", ev_xn["simplified_named"])
print()
print("Input (x_N keys):")
for k, v in ev_xn["input_normalized"].items():
    print(f"  {k:<6} = {v}")
print()
print(f"Output  : {ev_xn['output']}")
print(f"Finite  : {ev_xn['is_finite']}")
print(f"Warnings: {ev_xn['warnings'] or 'none'}")

Expression (simplified): x_10**3*x_7 + x_14 - x_7
Expression (named)     : MutPred_score**3*MutationTaster_score + VEST4_score - MutationTaster_score

Input (x_N keys):
  x_10   = 0.75
  x_7    = 0.82
  x_14   = 0.61

Output  : 0.1359375
Finite  : True
Warnings: none


In [4]:
# --- using display names ---
ev_named = evaluate_expression_at_input(EXPR, SAMPLE_INPUT_NAMED, VAR_MAP)
print("Output with display-name keys:", ev_named["output"])
print("Named input dict:", ev_named["input_named"])

Output with display-name keys: 0.1359375
Named input dict: {'MutPred_score': 0.75, 'MutationTaster_score': 0.82, 'VEST4_score': 0.61}


In [5]:
# --- singularity: x_10 = 0.5 makes the denominator zero ---
ev_sing = evaluate_expression_at_input(
    EXPR,
    {"x_10": 0.5, "x_7": 0.82, "x_14": 0.61},
    VAR_MAP,
)
print(f"Output  : {ev_sing['output']}")
print(f"Finite  : {ev_sing['is_finite']}")
print(f"Warnings: {ev_sing['warnings']}")

Output  : -0.10749999999999993
Finite  : True
Warnings: []


In [6]:
# --- missing variable: x_14 not provided ---
ev_miss = evaluate_expression_at_input(
    EXPR,
    {"x_10": 0.75, "x_7": 0.82},
    VAR_MAP,
)
print(f"Output  : {ev_miss['output']}")
print(f"Missing : {ev_miss['variables_not_in_input']}")
print(f"Warnings: {ev_miss['warnings']}")

Output  : None
Missing : ['x_14']
Warnings: ["Variables present in expression but missing from input: ['x_14']"]


## 4. `generate_local_facts`

Wraps `evaluate_expression_at_input` and assembles a structured local report

Extra options:
- **`model_context`** — free-form metadata (model source, domain annotations, target variable)  
  Kept strictly separate from computed facts; clearly labeled in the prompt as user-provided
- **`provided_output`** — if given, the computed output is compared against it and the  
  absolute discrepancy is reported. No attempt is made to explain the discrepancy

In [7]:
import pprint

# Basic: no context, no provided output
facts_basic = generate_local_facts(EXPR, SAMPLE_INPUT_XN, VAR_MAP)
print("=== basic (no context, no provided output) ===")
pprint.pprint(facts_basic)

=== basic (no context, no provided output) ===
{'expression': {'original': '((x_10**3 * x_7) + (x_14 - x_7))',
                'simplified': 'x_10**3*x_7 + x_14 - x_7',
                'simplified_named': 'MutPred_score**3*MutationTaster_score + '
                                    'VEST4_score - MutationTaster_score'},
 'input': {'values_named': {'MutPred_score': 0.75,
                            'MutationTaster_score': 0.82,
                            'VEST4_score': 0.61},
           'values_normalized': {'x_10': 0.75, 'x_14': 0.61, 'x_7': 0.82},
           'variables_in_expr': ['x_10', 'x_14', 'x_7'],
           'variables_not_in_input': []},
 'model_context': {},
 'output': {'computed': 0.1359375, 'is_finite': True},
 'output_comparison': None,
 'warnings': []}


In [8]:
# Full: with model_context and provided_output
facts_full = generate_local_facts(
    EXPR,
    SAMPLE_INPUT_XN,
    VAR_MAP,
    model_context=MODEL_CONTEXT,
    provided_output=PROVIDED_OUTPUT,
)
print("=== full (with context and provided output) ===")
pprint.pprint(facts_full)

=== full (with context and provided output) ===
{'expression': {'original': '((x_10**3 * x_7) + (x_14 - x_7))',
                'simplified': 'x_10**3*x_7 + x_14 - x_7',
                'simplified_named': 'MutPred_score**3*MutationTaster_score + '
                                    'VEST4_score - MutationTaster_score'},
 'input': {'values_named': {'MutPred_score': 0.75,
                            'MutationTaster_score': 0.82,
                            'VEST4_score': 0.61},
           'values_normalized': {'x_10': 0.75, 'x_14': 0.61, 'x_7': 0.82},
           'variables_in_expr': ['x_10', 'x_14', 'x_7'],
           'variables_not_in_input': []},
 'model_context': {'domain_annotations': {'MutPred_score': 'Probability of '
                                                           'pathogenicity '
                                                           'predicted by '
                                                           'MutPred2',
                                          'M

In [9]:
# Edge case: singularity, output_comparison is skipped
facts_sing = generate_local_facts(
    EXPR,
    {"x_10": 0.5, "x_7": 0.82, "x_14": 0.61},
    VAR_MAP,
    provided_output=4.1,
)
print("output         :", facts_sing["output"])
print("output_comparison:", facts_sing["output_comparison"])
print("warnings       :", facts_sing["warnings"])

output         : {'computed': -0.10749999999999993, 'is_finite': True}
output_comparison: {'computed': -0.10749999999999993, 'provided': 4.1, 'absolute_discrepancy': 4.2075}
warnings       : []


## 5. `generate_local_plain_language_prompt`

Converts `local_facts` into a prompt ready to be sent to a language model.

The prompt:
- separates **VERIFIED FACTS** (computed) from **USER-PROVIDED CONTEXT** (metadata)
- instructs the LLM not to infer causality, domain meaning, or feature importance
- instructs the LLM not to make local sensitivity claims
- instructs the LLM to report non-finite outputs and discrepancies factually, without interpretation

No LLM call is made here.

In [10]:
prompt = generate_local_plain_language_prompt(facts_full)
print(prompt)

You are a scientific writing assistant explaining a single prediction made by a symbolic regression model.

STRICT RULES:
1. Use only the facts in VERIFIED FACTS and the annotations in USER-PROVIDED CONTEXT.
2. Clearly distinguish between computed facts (VERIFIED FACTS) and user-provided annotations (USER-PROVIDED CONTEXT). Do not treat context as computed.
3. Do not infer causality or real-world reasons. You may describe how the expression is evaluated on this input, but do not claim that any variable caused the output.
4. Do not infer domain meaning from variable names unless explicitly given in USER-PROVIDED CONTEXT.
5. Do not make feature importance claims — do not say which input drove the output or contributed most.
6. Do not make local sensitivity claims — do not say what would happen if an input changed.
7. Do not mention variables not present in 'allowed_variable_names'.
8. If the output is non-finite, state that explicitly and do not attempt to interpret it.
9. If an output d